In [1]:
import sys
import os
import numpy as np
import time
import MDAnalysis as mda
from MDAnalysis.analysis import align
from MDAnalysis.analysis import distances

import matplotlib.pyplot as plt


/home/jonathan/anaconda3/envs/serpents/lib/python3.11/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:
#Chatgpt prompt:
"""write a python function that takes as its arguments a path to a pdb file, an mdanalysis atomgroup, and a list of numbers equal in length to the atom group, 
and saves a copy of the pdb file with the b factors of the atoms in the atom group set equal to the numbers in the list. 
Match atoms using residue names and numbers since the indices do not match between the atomgroup and pdb file"""


def write_bfactors_by_residue_match(pdb_path, atomgroup, values, output_pdb=None):
    """
    Write a copy of a PDB file with B-factors set for atoms matching an AtomGroup.
    Matching is done using (resname, resid, atom name), NOT indices.

    Parameters
    ----------
    pdb_path : str
        Path to input PDB file.
    atomgroup : MDAnalysis AtomGroup
        AtomGroup derived from (possibly different) structure.
    values : array-like
        Numbers equal in length to atomgroup.
    output_pdb : str
        Path to output PDB file.
    """

    if output_pdb is None:
        base = os.path.splitext(pdb_path)[0]
        output_pdb = base + "_bfactored.pdb"

    values = np.asarray(values)

    if len(atomgroup) != len(values):
        raise ValueError("Length of values must equal AtomGroup length.")

    # Load fresh universe from PDB to preserve original ordering
    u = mda.Universe(pdb_path)

    # Ensure tempfactors exist
    if not hasattr(u.atoms, "tempfactors"):
        u.add_TopologyAttr("tempfactors")

    # Start from existing B-factors (or zeros)
    try:
        b = u.atoms.tempfactors.copy()
    except AttributeError:
        b = np.zeros(len(u.atoms))

    # Build lookup dictionary from PDB universe
    pdb_lookup = {}
    for atom in u.atoms:
        key = (atom.resname, atom.resid, atom.name)
        pdb_lookup[key] = atom.index

    # Assign B-factors by matching keys
    for atom, value in zip(atomgroup, values):
        key = (atom.resname, atom.resid, atom.name)

        if key not in pdb_lookup:
            raise ValueError(
                f"Atom not found in PDB: resname={atom.resname}, "
                f"resid={atom.resid}, name={atom.name}"
            )

        pdb_index = pdb_lookup[key]
        b[pdb_index] = value

    # Reassign full array (required by MDAnalysis)
    u.atoms.tempfactors = b

    # Write output
    with mda.Writer(output_pdb, multiframe=False) as W:
        W.write(u.atoms)


In [3]:
def tmd_query():
    segment_resis = [[77, 149], [192, 245], [298, 362], [988, 1034], [857, 889], [900, 942], [1094, 1154]]
    #print("color blue, " + " or ".join([f"resi {sr[0]}-{sr[1]}" for sr in segment_resis]))

    segment_resis_all = [i for sr in segment_resis for i in range(sr[0], sr[1]+1)]
    query = " or ".join([f"resid {sr}" for sr in segment_resis_all])

    #indices = frame.top.select(f"protein and ({query})")

    # print_indices = frame.top.select(f"protein and name CA and ({query})")
    # print("+".join([str(i+1) for i in print_indices]))
    return f"protein and ({query})"

def binding_site_query():
    segment_resis = [[221, 245], [298, 328], [857, 886], [911, 942]]
    #print("color blue, " + " or ".join([f"resi {sr[0]}-{sr[1]}" for sr in segment_resis]))

    segment_resis_all = [i for sr in segment_resis for i in range(sr[0], sr[1]+1)]
    query = " or ".join([f"resid {sr}" for sr in segment_resis_all])

    #indices = frame.top.select(f"protein and ({query})")

    # print_indices = frame.top.select(f"protein and name CA and ({query})")
    # print("+".join([str(i+1) for i in print_indices]))
    return f"protein and ({query})"

#binding_site_query()

In [4]:
def main():
    # ============================
    # LOAD TRAJECTORY
    # ============================

    ref_frame_path_x01 = '/media/X01Raid01/Data_Backup/home/csheen/cftr-project/wstp_cftr_1_degrabo/bstates/input/min.gro'
    ref_frame_path = '/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.gro'

    ref_frame = mda.Universe(ref_frame_path)

    gro_file = "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.gro"
    xtc_file = "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/001913-000187-trj-pbcmol-centered-tmd-rot-s10.xtc"

    # Frame index to analyze
    frame_index = -1

    u = mda.Universe(gro_file, xtc_file)

    u.trajectory[frame_index]

    align.AlignTraj(u, ref_frame, select=f"{tmd_query()} and name CA")

    return u
    
u = main()

print(u.trajectory.n_frames)

573


In [5]:
def contacts_bin(group_1, group_2, cutoff=5.0):
    dists = distances.distance_array(group_1, group_2, box=u.dimensions)
    #print(dists.shape)
    contacts = np.where(dists < cutoff, 1, 0)
    contact_bin = np.where(np.sum(contacts, axis=1) > 0, 1, 0)

    return contact_bin

In [ ]:
# #!/usr/bin/env python3

def select_heavy_atoms(u):
    """Select heavy atoms based on a given selection string."""

    #print(u.trajectory.n_frames)

    # ============================
    # SELECTION PARAMETERS
    # ============================

    # List of protein residue numbers (edit as needed)
    #protein_resids = [10, 25, 48]   # <-- USER FILLS THIS
    protein_atoms = f"{tmd_query()} and not name H*"
    #protein_atoms = "resid 207 and resname GLN and not name H*"

    # Ligand residue name
    ligand_resname = "LJP"          # <-- USER FILLS THIS

    #not actually phosphates anymore
    # Lipid phosphate atom name (common choices: "P", "PO4", etc.)
    phosphate_selection = "resname PC and name N31" #"(resname PA or resname OL) and name C12"  #"resname PC and name P31"

    #membrane_core_buffer = 0


    # ============================
    # PROTEIN HEAVY ATOMS
    # ============================

    #resid_string = " ".join(str(r) for r in protein_resids)

    protein_sel = u.select_atoms(protein_atoms)

    # ============================
    # LIGAND HEAVY ATOMS
    # ============================

    ligand_sel = u.select_atoms(
        f"resname {ligand_resname} and not name H*"
    )

    # ============================
    # MEMBRANE PHOSPHATES
    # ============================

    phosphates = u.select_atoms(phosphate_selection)

    if len(phosphates) == 0:
        raise ValueError("No phosphate atoms found. Check lipid resnames and phosphate atom name.")

    #print(phosphates.positions.shape)

    z_positions = phosphates.positions[:, 2]
    z_mean = np.mean(z_positions)

    # Separate into upper and lower leaflets
    upper_leaflet = phosphates[z_positions > z_mean]
    lower_leaflet = phosphates[z_positions <= z_mean]

    if len(upper_leaflet) == 0 or len(lower_leaflet) == 0:
        raise ValueError("Leaflet separation failed.")

    z_upper_avg = np.mean(upper_leaflet.positions[:, 2])
    z_lower_avg = np.mean(lower_leaflet.positions[:, 2])

    z_min = min(z_upper_avg, z_lower_avg)
    z_max = max(z_upper_avg, z_lower_avg)

    # ============================
    # WATERS INSIDE MEMBRANE
    # ============================

    waters = u.select_atoms("resname TP3 and name O")

    water_z = waters.positions[:, 2]

    inside_mask = (water_z > z_min-4) & (water_z < z_max+4)
    waters_inside = waters[inside_mask]

    #print(np.where(inside_mask))
    #water_in_inds = np.where(inside_mask)[0]
    #print("color blue, index " + "+".join([str(i+1) for i in waters_inside.indices]))
    # ============================
    # OUTPUT SUMMARY
    # ============================

    #print(f"Protein heavy atoms: {len(protein_sel)}")
    #print(f"Ligand heavy atoms: {len(ligand_sel)}")
    #print(f"Waters inside membrane (heavy atoms): {len(waters_inside)}")

    # water_protein_dists = distances.distance_array(protein_sel,
    #                          waters_inside,
    #                          box=u.dimensions)

    # water_protein_contacts = np.where(water_protein_dists < 5, 1, 0)
    # protein_water_contact_bin = np.where(np.sum(water_protein_contacts, axis=1) > 0, 1, 0)

    #water contacts are averaged across waters so we get a 1d contact array
    protein_water_contacts = contacts_bin(protein_sel, waters_inside, cutoff=5.0)
    ligand_water_contacts = contacts_bin(ligand_sel, waters_inside, cutoff=5.0)

    #protein-ligand contacts are not averaged so we get a 2d contact array
    protein_ligand_dists = distances.distance_array(protein_sel,
                             ligand_sel,
                             box=u.dimensions)

    protein_ligand_contacts = np.where(protein_ligand_dists < 5, 1, 0)

    # water_protein_contacts = contacts.Contacts(u, select=(protein_sel, waters_inside), refgroup=(protein_sel, waters_inside), radius=4.5)
    # water_protein_contacts.run()
    # print(water_protein_contacts.results)
    # Example: save indices if desired
    # np.save("protein_indices.npy", protein_sel.indices)
    # np.save("ligand_indices.npy", ligand_sel.indices)
    # np.save("waters_inside_indices.npy", waters_inside.indices)

    protein_sel_2 = u.select_atoms(f"{protein_atoms} and name CA")
    return (waters_inside.positions, protein_water_contacts, ligand_water_contacts, protein_ligand_contacts, protein_sel_2.positions), protein_sel, ligand_sel

#wat_pos, pwc, lwc, plc, prot_sel, lig_sel = select_heavy_atoms(u)

In [7]:
observables_allwalkers = []

n_frames = u.trajectory.n_frames
#n_frames = 20

print(f"{n_frames} frames")
t0 = time.time()
for fri in range(n_frames):
    if fri%10 == 0:
        print(fri)
        t2 = time.time()
        print(t2-t0)

    u.trajectory[fri]
    observables, prot_sel, lig_sel = select_heavy_atoms(u)

    observables_allwalkers.append(observables)

t1 = time.time()
print(t1-t0)

output = []
n_observables = 5
for i_obs in range(n_observables):

    values = [o[i_obs] for o in observables_allwalkers if o is not None]

    #THIS CODE IS NOT GENERAL; IT DEPENDS ON THE DETAILS OF THE IMPORTED main() METHOD    
    if i_obs == 0:
        output.append(np.concatenate(values))
    else:
        output.append(np.stack(values))



573 frames
0
4.887580871582031e-05
10
5.068920135498047
20
10.2802734375
30
15.503491878509521
40
20.595487594604492
50
25.65570044517517
60
30.716980695724487
70
35.80101680755615
80
40.843921422958374
90
45.98089933395386
100
51.20189952850342
110
56.426856994628906
120
61.600202798843384
130
66.76424813270569
140
72.03874778747559
150
77.22242164611816
160
82.45214009284973
170
87.65624213218689
180
93.03706884384155
190
98.3499686717987
200
103.58558583259583
210
108.77659773826599
220
114.0396523475647
230
119.2226345539093
240
124.42729377746582
250
129.5500283241272
260
134.58055067062378
270
139.64075994491577
280
144.6650207042694
290
149.67213797569275
300
154.70998287200928
310
159.8385112285614
320
165.20207500457764
330
170.43283677101135
340
175.7014675140381
350
181.03600931167603
360
186.40227556228638
370
191.70619225502014
380
196.76040077209473
390
201.8100597858429
400
207.01760697364807
410
212.1589298248291
420
217.41987943649292
430
222.74082612991333
440
227.979

In [8]:
import numpy as np
import plotly.graph_objects as go

# Example large dataset
# N = 200000
# coords = np.random.randn(N,3)

# x = coords[:,0]
# y = coords[:,1]
# z = coords[:,2]
interval=50
x = output[0][::interval,0]
y = output[0][::interval,1]
z = output[0][::interval,2]

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="markers",
        hoverinfo="skip",
        marker=dict(
            size=2,
            opacity=0.7,
            color=z,          # GPU color mapping
            colorscale="Viridis"
        )
    )
)

xp = output[4][0,:,0]#.flatten()
yp = output[4][0,:,1]#.flatten()
zp = output[4][0,:,2]#.flatten()

print(xp.shape)

fig.add_trace(
    go.Scatter3d(
        x=xp,
        y=yp,
        z=zp,
        mode="markers",
        hoverinfo="skip",
        marker=dict(
            size=2,
            opacity=0.7,
            color="black"
        )
    )
)

fig.update_layout(
    title="Large Interactive 3D Scatter",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    margin=dict(l=0,r=0,b=0,t=40),
    scene_camera=dict(
            eye=dict(x=3, y=3, z=3)
        )

)

fig.show(config={"scrollZoom": True})

#fig.show()

(376,)


In [ ]:
print(output[0].shape)
plt.hist2d(output[0][:,0], output[0][:,1], bins = (20,20), range=((25,75),(25,75)))
plt.show()
plt.hist2d(output[0][:,0], output[0][:,2], bins = (20,20), range=((25,75),(70,95)))
plt.show()
plt.hist2d(output[0][:,1], output[0][:,2], bins = (20,20), range=((25,75),(70,95)))
#plt.show()


In [ ]:
observables, prot_sel, lig_sel = select_heavy_atoms(u)

In [ ]:
plt.plot(np.mean())

In [ ]:
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", prot_sel, pwc,
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_pwc.pdb")
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", lig_sel, lwc,
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_lwc.pdb")


In [ ]:
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", lig_sel, np.sum(plc, axis=0),
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_plc_lig.pdb")
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", prot_sel, np.sum(plc, axis=1),
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_plc_prot.pdb")

In [ ]:
(output[0] == output_r1[0]).all()

In [ ]:
output_r1 = output

In [ ]:
tmd_query()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Generate example data
n = 200
x = np.random.randn(n)
y = np.random.randn(n)
z = np.random.randn(n)

# Create 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode='markers',
            marker=dict(
                size=5,
                color=z,      # color by z value
                colorscale='Viridis',
                opacity=0.8
            )
        )
    ]
)

fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    title="Interactive 3D Scatter"
)

fig.show()